In [104]:
import pandas as pd

Goal: We want to maximize amount of money saved by Big G. A full derate costs around \\$4000 for towing, plus the cost of
repairs. The cost of a false positive prediction is about \\$500 due to having the truck oﬀ the road and serviced unnecessarily. Thus, we'll want to optimize our model for recall.

We want to attempt to predict an idle derate at least **2 hours** before it occurs.

In [105]:
faults = pd.read_csv("../data/J1939Faults.csv")

# Dropping columns that are not relevant
faults_clean = faults.drop(columns=['ecuSoftwareVersion', 'ecuSerialNumber', 'ecuModel', 'ecuMake', 'ecuSource', 'MCTNumber', 'ESS_Id', 'actionDescription', 'faultValue'])

# Concatenating spn and fmi into a new column
faults_clean['spn-fmi'] = faults_clean['spn'].astype(str) + "-" + faults_clean['fmi'].astype(str)

faults_clean.head()

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_28244/1086016911.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  faults = pd.read_csv("../data/J1939Faults.csv")


,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,LocationTimeStamp,spn-fmi
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,2015-02-21 11:34:25.000,111-17
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,2015-02-21 11:35:10.000,629-12
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,2015-02-21 11:35:26.000,1807-2
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,2015-02-21 11:36:08.000,1807-2
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,2015-02-21 11:39:37.000,4364-17


Looking at the first record, here is a breakdown of the important values.

* ESS_Id, actionDescription, ecuSoftwareVersion, ecuSerialNumber, ecuModel, ecuMake, ecuSource, faultValue, and MCTNumber are unlikely to provide any predictive value.
* We can see the time of the event in the **EventTimeStamp** column. Note that this may be different from the **LocationTimeStamp** value, which indicates when the Latitude/Longitude values were recorded.
* The **spn** and **fmi** columns together indicate the type of fault, and there may be a description of that fault in the **eventDescription** column, although this column is sometimes missing.
* Faults are recorded when the light goes on and when it goes off, which is indicated by the **active** column, with True indicating the light turning on and False indicating turning off. The number of times the code has been set or unset is in the **faultValue** column, although this value can be unreliable. 
* Each truck has an identifier, the **EquipmentID** value.
* Each record can be linked to the on-board diagnostics data through the **RecordID** column.

Filtering out vehicles that are likely being serviced: (We'll come back to this)

Service stations are at (36.0666667, -86.4347222), (35.5883333, -86.4438888), and (36.1950, -83.174722)

In [106]:
# Future code here

In [107]:
diagnostics = pd.read_csv("../data/VehicleDiagnosticOnboardData.csv")

diagnostics.head()

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1


Pivoting diagnostic onboard data by fault ID so that it's easier to merge. 

In [108]:
diagnostics_pivoted = diagnostics.pivot(index='FaultId', columns='Name', values='Value')

In [109]:
diagnostics_pivoted.head()

Name,AcceleratorPedal,BarometricPressure,CruiseControlActive,CruiseControlSetSpeed,DistanceLtd,EngineCoolantTemperature,EngineLoad,EngineOilPressure,EngineOilTemperature,EngineRpm,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
FaultId,,,,,,,,,,,,,,,,,,,,,
1,0,14.21,False,66.48672,423178.7,100.4,11,0,96.74375,0,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


To merge diagnostic info with fault code info, we'll match up the **RecordID** and **FaultId**.

In [110]:
faults_full = pd.merge(faults_clean, diagnostics_pivoted, left_on='RecordID', right_on='FaultId', how='outer')

faults_full.head()

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


Creating target column:

Our plan will be to make 4 different target columns: \
* Derate in 0-2 hours
* Derate in 2-6 hours
* Derate in 6-12 hours
* Derate in 12-24 hours

Then, we'll fit our model on each of the 4 target columns and see which model is most predictive.

The idea is to give the vehicle operator enough warning so that they have enough time to drive to a service station. 0-2 hours of warning probably isn't enough time. As we increase the window larger and larger, we'll probably lose some predictive power. So we'll try to find the sweet spot by evaluating several different time windows.

In [111]:
# Creating a new column to mark whether the fault represents a full derate
# We'll use this when we apply .rolling later 
faults_full['derate_flag'] = ((faults_full["spn"] == 5246) & (faults_full["active"] == True)).astype('int')

In [112]:
# Converting EventTimeStamp to datetime 
faults_full['EventTimeStamp']= pd.to_datetime(faults_full['EventTimeStamp'])

In [113]:
# Running into an issue with duplicated EquipmentID/EventTimeStamp. Here's an example:
faults_full[(faults_full['EquipmentID']==301) & (faults_full['EventTimeStamp']=="2016-03-20 05:08:11")]

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,...,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure,derate_flag
413901,422168,2016-03-20 05:08:11,Incorrect Data Engine Exhaust Back Pressure Re...,649,2,True,1,301,36.067129,-86.434814,...,True,46.4,18431,True,NaN,0,NaN,0,1.16,0
413902,422169,2016-03-20 05:08:11,NaN,5625,16,True,1,301,36.067129,-86.434814,...,True,46.4,18431,True,NaN,0,NaN,0,1.16,0
413903,422170,2016-03-20 05:08:11,Error in System Engine Exhaust Gas Recirculati...,3821,11,True,1,301,36.067129,-86.434814,...,True,46.4,18431,True,NaN,0,NaN,0,1.16,0


In [114]:
# The .rolling method can't handle the duplicated EquipmentID and EventTimeStamp columns.
# We'll need to de-duplicate before we apply .rolling
# By taking the max of 'derate_flag', we're ensuring that for each EquipmentID/EventTimeStamp combination, 
#       we're capturing whether at least one derate occurred for that vehicle at that time (if multiple faults occurred simultaneously)
faults_dedup = faults_full.groupby(['EquipmentID', 'EventTimeStamp']).max('derate_flag').reset_index()

In [85]:
faults_dedup.head()

,EquipmentID,EventTimeStamp,RecordID,spn,fmi,active,activeTransitionCount,Latitude,Longitude,derate_flag
0,301,2015-04-26 06:16:16,34467,190,0,True,126,36.066527,-86.435277,0
1,301,2015-04-28 05:29:21,36192,190,0,True,126,36.066666,-86.435000,0
2,301,2015-05-10 07:11:34,48298,639,2,True,127,36.067083,-86.433842,0
3,301,2015-05-10 07:59:25,48325,639,2,False,127,35.586805,-86.443472,0
4,301,2015-05-11 13:11:20,49415,639,2,True,127,36.189398,-82.795601,0


In [115]:
# By sorting the EventTimeStamp in descending order, we're ensuring that we are rolling forwards in time (since .rolling looks at preceding rows)
# Then we're applying the .rolling method with a 2 hour time window and taking the max derate_flag.
# The new 'derate_2_hr' column is indicating whether or not a derate happened in the next 2 hours for the given vehicle.
intermediate = faults_dedup.sort_values(by='EventTimeStamp', ascending=False).groupby('EquipmentID').rolling(window='2h', on='EventTimeStamp')['derate_flag'].max().reset_index().rename(columns={'derate_flag':'derate_2_hr'})

In [116]:
# Next, we'll merge the new 'derate_2_hr' column into the original DataFrame, mergine on EventTimeStamp and EquipmentID
# Any duplicate EventTimeStamp/EquipmentID combinations will have the same value in the target column
faults_full = pd.merge(intermediate, faults_full, on=['EventTimeStamp','EquipmentID'], how='right')

In [121]:
faults_full.head()

,EquipmentID,EventTimeStamp,derate_2_hr,RecordID,eventDescription,spn,fmi,active,activeTransitionCount,Latitude,...,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure,derate_flag
0,1439,2015-02-21 10:47:13,0.0,1,Low (Severity Low) Engine Coolant Level,111,17,True,2,38.857638,...,False,78.8,1023,True,NaN,0,3276.75,NaN,0,0
1,1439,2015-02-21 11:34:34,0.0,2,NaN,629,12,True,127,38.857638,...,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1369,2015-02-21 11:35:31,0.0,3,Incorrect Data Steering Wheel Angle,1807,2,False,127,41.421250,...,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1369,2015-02-21 11:35:33,0.0,4,Incorrect Data Steering Wheel Angle,1807,2,True,127,41.421018,...,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1674,2015-02-21 11:39:41,0.0,5,NaN,4364,17,False,2,38.416481,...,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN,0


We can get a little bit more information about the different fault codes from the Service Fault Codes spreadsheet. We'll come back to this information later to see if we think it would be a helpful addition to our model. The problem is there can be multiple rows for a given fault code, so it's unclear which information should be used. As a stretch goal, we could analyze the algorithm description or lamp color to bucket issues into mild/moderate/severe categories.

In [27]:
sfc = pd.read_excel("../data/Service Fault Codes_1_0_0_167.xlsx")
sfc.head()

/home/michael/anaconda3/lib/python3.9/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
1,Y,112,167,Not Mapped,20,128,7,635,7,Not Mapped,Red,Stop / Shutdown,Engine Timing Actuator Driver Circuit - Mechan...,Mechanical failure in the engine timing actuat...
2,Y,113,167,Not Mapped,20,128,3,635,3,Not Mapped,Amber,Warning,Engine Timing Actuator Driver Circuit - Voltag...,High signal voltage detected at the engine tim...
3,Y,114,167,Not Mapped,20,128,4,635,4,Not Mapped,Amber,Warning,Engine Timing Actuator Driver Circuit - Voltag...,Low voltage detected at the engine timing actu...
4,Y,115,167,190,Not Mapped,Not Mapped,2,612,2,P0008,Red,Stop / Shutdown,Engine Magnetic Speed/Position Lost Both of Tw...,The ECM has detected that the primary and back...


For a large number of fault codes, there are multiple records. For example, if we look at the rows for the first fault in our dataset, we see that there are two rows.

In [28]:
(
    sfc
    .loc[sfc['SPN'] == 111]
    .loc[sfc['J1939 FMI'] == 17]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
1589,Y,2448,167,111,Not Mapped,Not Mapped,1,111,17,P2560,Maintenance,Maintenance,Coolant Level - Data Valid But Below Normal Op...,Low engine coolant level detected.
3540,Y,5167,167,Not Mapped,Not Mapped,Not Mapped,1,111,17,Not Mapped,Amber,Warning,Coolant Level - Data Valid But Below Normal Op...,NaN


Or even more.

In [29]:
(
    sfc
    .loc[sfc['SPN'] == 629]
    .loc[sfc['J1939 FMI'] == 12]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
180,Y,343,167,Not Mapped,254,0,12,629,12,P0607,Amber,Warning,Engine Control Module Warning Internal Hardwar...,ECM power supply errors have been detected.
689,Y,1116,167,Not Mapped,254,0,12,629,12,Not Mapped,Amber,Warning,Engine Control Module Critical Internal Failur...,ECM Internal failure has occurred.
854,Y,1388,167,Not Mapped,254,0,12,629,12,Not Mapped,NaN,NaN,Engine Control Module Data Lost - Bad Intellig...,The ECM data has been lost.
1019,Y,1597,167,Not Mapped,254,0,12,629,12,Not Mapped,Maintenance,Maintenance,Engine Control Module Critical Internal Failur...,The ECM has occurred an internal failure.
